In [0]:
%sql
select * from all_academy
-- where name = "Fredra Wealthall"

In [0]:
applicants_df = spark.table("all_applicants")
applicants_df.show()

# applicants_df.printSchema()

In [0]:
from pyspark.sql.functions import *

candidate_df = (applicants_df.withColumn("email",lower(trim(col("email")))).dropDuplicates()) #

candidate_df.show()

In [0]:
academy_df = spark.table("all_academy")
academy_df.show()



In [0]:
competency_columns = [c for c in academy_df.columns if "_W" in c]
competencies = sorted(set(c.split("_")[0]for c in competency_columns))
competency_df = spark.createDataFrame([(c,) for c in competency_columns],["competency_name"])
print(competencies)


In [0]:
competency_df = spark.createDataFrame([(c,) for c in competencies],["competency_name"])
competency_df.show()

In [0]:
weeks = sorted(set(int(c.split("_W")[1])for c in competency_columns))
print(weeks)

In [0]:

week_df = spark.createDataFrame([(w,) for w in weeks],["week"])
week_df.show()


In [0]:
trainer_df = (academy_df.select("trainer").distinct())
trainer_df.show()

In [0]:
from pyspark.sql.functions import row_number  
from pyspark.sql.window import Window 

trainer_df = trainer_df.withColumn("trainer_id",row_number().over(Window.orderBy("trainer")))
trainer_df = trainer_df.select("trainer_id","trainer")
trainer_df.show()


In [0]:
week = [(i,)for i in range(1,11)]
week_df = spark.createDataFrame(week,["week"])
week_df.show()

creating a weekly review 

In [0]:
from pyspark.sql.functions import lit
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

academy_df = spark.table("all_academy")

# Get unique trainee/trainer combinations
base_df = (academy_df.select("name", "trainer").distinct())
# Create one row per trainee per week
weekly_review_df = None

# base_df.show()
for week in weeks:
    temp_df = base_df.withColumn("week",lit(week))
    if weekly_review_df is None:
        weekly_review_df = temp_df
    else:
        weekly_review_df = weekly_review_df.union(temp_df)

weekly_review_df = weekly_review_df.withColumn("review_id",row_number().over(Window.orderBy("name", "week")))

# Reorder columns
weekly_review_df = weekly_review_df.select(
    "review_id",
    "name",
    "trainer",
    "week"
)

weekly_review_df.show()



In [0]:
weekly_review_df.count()